# Module score 

In [1]:
options(warn=-1)

In [2]:
library_load <- suppressMessages(
    
    suppressWarnings(
        
        list(
        
            # Seurat 
            library(Seurat), 

            # UCELL 
            library(UCell), 
            
            # Data 
            library(tidyverse), 
            library(data.table), 
            library(reactable), 

            # Plotting 
            library(ggplot2),  

            # Pyhton compatibility
            library(reticulate)

        )
    )
)

In [3]:
# Configure reticulate 
use_condaenv(condaenv="p.3.10.16-FD20250213HSCT", conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)

In [4]:
random_seed <- 42
set.seed(random_seed)

In [5]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20250213HSCT")

In [6]:
# Plotting Theme
source("plotting_global.R")
ggplot2::theme_set(theme_global_set(size_select=1)) # From project global source()

# Parameter 

In [7]:
compute <- FALSE

In [8]:
# Import Seurat object
so <- readRDS("data/object/pp.rds")

# TRM17

In [9]:
# TRM1/TRM17
TRM17_genes <- c("CCR6", "IL7R", "MAF", "RORC")
TRM1_genes <- c("TBX21", "ZNF683", "IFNG", "CXCR3")

TRM17_vs_TRM1_genes <- c("CCR6+", "IL7R+", "MAF+", "RORC+", "TBX21-", "ZNF683-", "IFNG-", "CXCR3-")
TRM1_vs_TRM17_genes <- c("CCR6-", "IL7R-", "MAF-", "RORC-", "TBX21+", "ZNF683+", "IFNG+", "CXCR3+")

# T cell residency
TCM_genes <- c("SELL+", "CCR7+", "TCF7+", "CD27+", "MAL+", "S1PR1+", "CX3CR1+", "CD69-","PRDM1-","EZR-","LMNA-","VIM-","CXCR4-")
TRM_genes <- c("SELL-", "CCR7-", "TCF7-", "CD27-", "MAL-", "S1PR1-", "CX3CR1-", "CD69+","PRDM1+","EZR+","LMNA+","VIM+","CXCR4+")

In [10]:
if (compute) {

    so <- UCell::AddModuleScore_UCell(so, features=list(TRM17_ucell=TRM17_genes, TRM1_ucell=TRM1_genes, TRM17_vs_TRM1_ucell=TRM17_vs_TRM1_genes, TRM1_vs_TRM17_ucell=TRM1_vs_TRM17_genes, TRM_ucell=TRM_genes, TCM_ucell=TCM_genes), assay="RNA", slot="counts", name="")
    write.csv(so@meta.data[, c("TRM17_ucell", "TRM1_ucell", "TRM17_vs_TRM1_ucell", "TRM1_vs_TRM17_ucell", "TRM_ucell", "TCM_ucell")], "result/ms/ms.csv")
    
} else {

    ms <- read.csv("result/ms/ms.csv", row.names=1)
    so <- AddMetaData(so, ms)
    
}

# Score T17hi/lo subsets and save annotation 

In [11]:
t17hi_blood_thr <- 0.25
t17hi_skin_thr <- 0.25

In [12]:
so$t_cell_subset <- ifelse((so$TRM17_vs_TRM1_ucell>=t17hi_blood_thr & so$TRM1_ucell<=0.00 & so$hto_demux_class=="Blood") | (so$TRM17_vs_TRM1_ucell>=t17hi_skin_thr & so$TRM1_ucell<=0.00 & so$hto_demux_class=="Skin"), "T17hi", "T17lo")
so$t_cell_subset <- ifelse(so$celltype_low %in% c("T['h']*' N/CM'", "T['h']*' E/EM'", "T['h']*' RM'", "T['c']*' N/CM'", "T['c']*' E/EM'", "T['c']*' RM'"), so$t_cell_subset, NA)

In [13]:
so$t_cell_subset <- ifelse(so$celltype_low %in% c("T['h']*' N/CM'", "T['h']*' E/EM'", "T['h']*' RM'") & so$t_cell_subset=="T17hi", "Th17hi", ifelse(so$celltype_low %in% c("T['h']*' N/CM'", "T['h']*' E/EM'", "T['h']*' RM'") & so$t_cell_subset=="T17lo", "Th17lo", so$t_cell_subset))
so$t_cell_subset <- ifelse(so$celltype_low %in% c("T['c']*' N/CM'", "T['c']*' E/EM'", "T['c']*' RM'") & so$t_cell_subset=="T17hi", "Tc17hi", ifelse(so$celltype_low %in% c("T['c']*' N/CM'", "T['c']*' E/EM'", "T['c']*' RM'") & so$t_cell_subset=="T17lo", "Tc17lo", so$t_cell_subset))

In [14]:
so$t_cell_subset <- factor(so$t_cell_subset, levels=c("Th17lo", "Th17hi", "Tc17lo", "Tc17hi"))

In [15]:
write.csv(so@meta.data[, "t_cell_subset", drop=FALSE], "result/annotation/t_cell_subset.csv")